# Question1 Part 2 - Transformer Model for translation




In [ ]:
import pandas as pd
import re

# 1. Load the TSV
path = "Sentence pairs in English-Russian - 2025-11-29.tsv"
df = pd.read_csv(path, sep="\t", engine="python", on_bad_lines="skip", header=None)

# 2. Rename columns for clarity
df.columns = ["en_id", "en_sentence", "es_id", "es_sentence"] #was originally done in spanish, forgive the column names....
#we ideally want to compare the same english words...

# 3. Simple clean function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()               # remove leading/trailing spaces
    text = text.lower()               # lowercase
    text = re.sub(r"\s+", " ", text)  # collapse multiple spaces
    return text

df["en_clean"] = df["en_sentence"].apply(clean_text)
df["es_clean"] = df["es_sentence"].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

# First: train (80%) vs temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

# Then: validation (10%) vs test (10%) from temp (20% of original)
# 0.5 * 0.2 = 0.1 of the original data for each
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

print(len(df), "total")
print(len(train_df), "train")
print(len(val_df), "validation")
print(len(test_df), "test")

201040 total
160832 train
20104 validation
20104 test


In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import math
import numpy as np

In [ ]:
# Get lists of sentences
train_en = train_df["en_clean"].astype(str).tolist()
train_es = train_df["es_clean"].astype(str).tolist()

val_en = val_df["en_clean"].astype(str).tolist()
val_es = val_df["es_clean"].astype(str).tolist()

test_en = test_df["en_clean"].astype(str).tolist()
test_es = test_df["es_clean"].astype(str).tolist()

# Special tokens
PAD_TOKEN = "<pad>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

special_tokens = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
max_vocab_size = 20000

def build_vocab(texts, max_size=max_vocab_size):
    counter = Counter()
    for sent in texts:
        tokens = sent.split()
        counter.update(tokens)
    # most common
    most_common = counter.most_common(max_size - len(special_tokens))
    itos = special_tokens + [w for w, _ in most_common]
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

src_stoi, src_itos = build_vocab(train_en)  # English
tgt_stoi, tgt_itos = build_vocab(train_es)  # Spanish

PAD_IDX = src_stoi[PAD_TOKEN]  # same indices for both vocabs for special tokens
SOS_IDX = src_stoi[SOS_TOKEN]
EOS_IDX = src_stoi[EOS_TOKEN]
UNK_IDX = src_stoi[UNK_TOKEN]

tgt_PAD_IDX = tgt_stoi[PAD_TOKEN]
tgt_SOS_IDX = tgt_stoi[SOS_TOKEN]
tgt_EOS_IDX = tgt_stoi[EOS_TOKEN]
tgt_UNK_IDX = tgt_stoi[UNK_TOKEN]

src_vocab_size = len(src_itos)
tgt_vocab_size = len(tgt_itos)

print("SRC vocab size:", src_vocab_size)
print("TGT vocab size:", tgt_vocab_size)

SRC vocab size: 20000
TGT vocab size: 20000


In [ ]:
def encode_src(sentence):
    tokens = sentence.split()
    return [src_stoi.get(tok, UNK_IDX) for tok in tokens]

def encode_tgt(sentence):
    tokens = sentence.split()
    ids = [tgt_SOS_IDX] + [tgt_stoi.get(tok, tgt_UNK_IDX) for tok in tokens] + [tgt_EOS_IDX]
    return ids

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_ids = encode_src(self.src_texts[idx])
        tgt_ids = encode_tgt(self.tgt_texts[idx])
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)


def collate_fn(batch):
    """
    batch: list of (src_ids, tgt_ids) with variable lengths.
    Returns padded src, tgt_inp, tgt_out
    """
    src_seqs, tgt_seqs = zip(*batch)  # each is a list of 1D tensors

    # Build decoder input (without last token) and output (without first token)
    tgt_inp_seqs = [seq[:-1] for seq in tgt_seqs]
    tgt_out_seqs = [seq[1:] for seq in tgt_seqs]

    # Pad using pad_sequence
    src_padded = nn.utils.rnn.pad_sequence(src_seqs, batch_first=True, padding_value=PAD_IDX)
    tgt_inp_padded = nn.utils.rnn.pad_sequence(tgt_inp_seqs, batch_first=True, padding_value=tgt_PAD_IDX)
    tgt_out_padded = nn.utils.rnn.pad_sequence(tgt_out_seqs, batch_first=True, padding_value=tgt_PAD_IDX)

    return src_padded, tgt_inp_padded, tgt_out_padded

batch_size = 64

train_dataset = TranslationDataset(train_en, train_es)
val_dataset   = TranslationDataset(val_en, val_es)
test_dataset  = TranslationDataset(test_en, test_es)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return x

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward=512,
        dropout=0.1,
        max_len=500,
    ):
        super().__init__()

        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size, padding_idx=PAD_IDX)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size, padding_idx=tgt_PAD_IDX)

        self.positional_encoding = PositionalEncoding(emb_size, max_len=max_len)

        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=False,  # expects (seq, batch, d_model)
        )

        self.generator = nn.Linear(emb_size, tgt_vocab_size)

        self.emb_size = emb_size

    def generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.full((sz, sz), float("-inf")), diagonal=1)
        return mask

    def create_padding_mask(self, seq, pad_idx):
        # seq: (batch, seq_len)
        return seq == pad_idx  # True where padded

    def forward(self, src, tgt):
        """
        src: (batch, src_len)
        tgt: (batch, tgt_len) -> this is tgt_input
        """
        src_mask = None
        tgt_seq_len = tgt.size(1)
        tgt_mask = self.generate_square_subsequent_mask(tgt_seq_len).to(tgt.device)

        src_padding_mask = self.create_padding_mask(src, PAD_IDX)          # (batch, src_len)
        tgt_padding_mask = self.create_padding_mask(tgt, tgt_PAD_IDX)      # (batch, tgt_len)

        # Embeddings + positional encoding
        src_emb = self.src_tok_emb(src) * math.sqrt(self.emb_size)  # (batch, src_len, emb)
        tgt_emb = self.tgt_tok_emb(tgt) * math.sqrt(self.emb_size)

        src_emb = self.positional_encoding(src_emb)
        tgt_emb = self.positional_encoding(tgt_emb)

        # Transformer expects (seq_len, batch, emb)
        src_emb = src_emb.transpose(0, 1)
        tgt_emb = tgt_emb.transpose(0, 1)

        output = self.transformer(
            src_emb,
            tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )
        # (tgt_len, batch, emb) -> (batch, tgt_len, emb)
        output = output.transpose(0, 1)
        logits = self.generator(output)  # (batch, tgt_len, tgt_vocab_size)
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Smaller model hyperparameters
emb_size = 128       # was 256
nhead = 2            # was 4
num_encoder_layers = 2   # was 3 (or even 1)
num_decoder_layers = 2   # was 3 (or even 1)
ff_dim = 256         # was 512

model = Seq2SeqTransformer(
    num_encoder_layers=num_encoder_layers,
    num_decoder_layers=num_decoder_layers,
    emb_size=emb_size,
    nhead=nhead,
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    dim_feedforward=ff_dim,
    dropout=0.1,
    max_len=500,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tgt_PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)  # slightly higher LR


Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for src, tgt_inp, tgt_out in dataloader:
        src = src.to(device)         # (batch, src_len)
        tgt_inp = tgt_inp.to(device) # (batch, tgt_len)
        tgt_out = tgt_out.to(device) # (batch, tgt_len)

        optimizer.zero_grad()
        logits = model(src, tgt_inp)  # (batch, tgt_len, vocab_size)

        # reshape for loss: (batch*tgt_len, vocab_size) and (batch*tgt_len)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    for src, tgt_inp, tgt_out in dataloader:
        src = src.to(device)
        tgt_inp = tgt_inp.to(device)
        tgt_out = tgt_out.to(device)

        logits = model(src, tgt_inp)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )
        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
num_epochs = 5

for epoch in range(1, num_epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch:02d}: train loss = {train_loss:.4f}, val loss = {val_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch 01: train loss = 5.1415, val loss = 4.1023
Epoch 02: train loss = 3.9631, val loss = 3.4329
Epoch 03: train loss = 3.3692, val loss = 3.0218
Epoch 04: train loss = 2.9470, val loss = 2.7457
Epoch 05: train loss = 2.6313, val loss = 2.5503


In [ ]:
tgt_idx_to_token = {i: tok for i, tok in enumerate(tgt_itos)}

def greedy_decode(model, src_sentence, max_len=50):
    model.eval()
    src_ids = torch.tensor([encode_src(src_sentence)], dtype=torch.long).to(device)
    # start with <sos>
    tgt_ids = torch.tensor([[tgt_SOS_IDX]], dtype=torch.long).to(device)

    for _ in range(max_len):
        logits = model(src_ids, tgt_ids)  # (1, cur_len, vocab_size)
        next_token_logits = logits[0, -1, :]  # last time step
        next_token_id = next_token_logits.argmax(-1).item()

        # stop at <eos>
        if next_token_id == tgt_EOS_IDX:
            break

        tgt_ids = torch.cat(
            [tgt_ids, torch.tensor([[next_token_id]], dtype=torch.long, device=device)],
            dim=1
        )

    # convert ids to tokens (skip first <sos>)
    out_ids = tgt_ids[0].tolist()[1:]
    tokens = [tgt_idx_to_token[i] for i in out_ids if i not in [tgt_PAD_IDX, tgt_EOS_IDX]]
    return " ".join(tokens)

# Example:
print(greedy_decode(model, "how are you?"))

как дела?


## Comparing the two models

In [ ]:
df_compare = pd.read_csv('Part_1_results.csv', usecols=lambda x: x not in ['Unnamed: 0'])
df_compare["Transformer_Translation"] = None
df_compare["Blue_Score_Transformer"] = None
df_compare

,Original English,Predicted Russian,Actual Russian,Blue_Score_LSTM,Transformer_Translation,Blue_Score_Transformer
0,I'd never hire Tom.,я не никогда не слышал,я бы никогда не взял тома на работу,5.094697e-155,None,None
1,Tom hung up angrily.,том не,том со злостью повесил трубку,3.418292e-232,None,None
2,"When the captain commands, the crew must obey.",когда в в в,когда капитан отдаёт приказ команда должна под...,6.085166e-232,None,None
3,I'm not prepared to share this information wit...,я не не не не,я не готов делиться с вами этой информацией,4.603578e-155,None,None
4,"The older he got, the more famous he became.",в в в в,чем старше он становился тем большую известность,0.000000e+00,None,None
...,...,...,...,...,...,...
95,He knows who killed her.,он знает что тома,он знает кто её убил,7.422681e-155,None,None
96,It's his first day at school.,это в в в в,это его первый день в школе,1.186218e-231,None,None
97,I don't remember what his name was.,я не не что это не,я не помню как его звали,7.579654e-155,None,None
98,Do you want kids?,ты том в,вы хотите детей,0.000000e+00,None,None


In [ ]:
transformer_translations = []
for english_sentence in df_compare["Original English"]:

    cleaned_english_sentence = clean_text(english_sentence)
    translated_sentence = greedy_decode(model, cleaned_english_sentence)
    transformer_translations.append(translated_sentence)

df_compare["Transformer_Translation"] = transformer_translations

df_compare.head()

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


,Original English,Predicted Russian,Actual Russian,Blue_Score_LSTM,Transformer_Translation,Blue_Score_Transformer
0,I'd never hire Tom.,я не никогда не слышал,я бы никогда не взял тома на работу,5.094697e-155,я бы никогда не <unk> тома.,None
1,Tom hung up angrily.,том не,том со злостью повесил трубку,3.418292e-232,том встал <unk>,None
2,"When the captain commands, the crew must obey.",когда в в в,когда капитан отдаёт приказ команда должна под...,6.085166e-232,когда <unk> <unk> <unk> <unk> <unk> <unk> <unk...,None
3,I'm not prepared to share this information wit...,я не не не не,я не готов делиться с вами этой информацией,4.603578e-155,я не привык с тобой <unk>,None
4,"The older he got, the more famous he became.",в в в в,чем старше он становился тем большую известность,0.000000e+00,он старше <unk> <unk>,None


In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method1

for index, row in df_compare.iterrows():
    # The 'Original English' is the source, 'Actual Russian' is the reference.
    # The 'Transformer_Translation' is the candidate translation from the model.
    actual_russian = row["Actual Russian"]
    predicted_russian_transformer = row["Transformer_Translation"]

    # Prepare for BLEU score calculation
    # Reference needs to be a list of lists of tokenized words
    # Candidate needs to be a list of tokenized words
    reference_tokens = [actual_russian.split()]
    candidate_tokens = predicted_russian_transformer.split()

    # Calculate BLEU score
    score = sentence_bleu(reference_tokens, candidate_tokens, smoothing_function=smoothie)

    # Directly assign the score to the DataFrame column
    df_compare.loc[index, "Blue_Score_Transformer"] = score

df_compare

,Original English,Predicted Russian,Actual Russian,Blue_Score_LSTM,Transformer_Translation,Blue_Score_Transformer
0,I'd never hire Tom.,я не никогда не слышал,я бы никогда не взял тома на работу,5.094697e-155,я бы никогда не <unk> тома.,0.364093
1,Tom hung up angrily.,том не,том со злостью повесил трубку,3.418292e-232,том встал <unk>,0.058335
2,"When the captain commands, the crew must obey.",когда в в в,когда капитан отдаёт приказ команда должна под...,6.085166e-232,когда <unk> <unk> <unk> <unk> <unk> <unk> <unk...,0.004874
3,I'm not prepared to share this information wit...,я не не не не,я не готов делиться с вами этой информацией,4.603578e-155,я не привык с тобой <unk>,0.068460
4,"The older he got, the more famous he became.",в в в в,чем старше он становился тем большую известность,0.000000e+00,он старше <unk> <unk>,0.045132
...,...,...,...,...,...,...
95,He knows who killed her.,он знает что тома,он знает кто её убил,7.422681e-155,"он знает, кто убил её.",0.070711
96,It's his first day at school.,это в в в в,это его первый день в школе,1.186218e-231,это первый раз в школе.,0.057893
97,I don't remember what his name was.,я не не что это не,я не помню как его звали,7.579654e-155,"я не помню, что его имя",0.095544
98,Do you want kids?,ты том в,вы хотите детей,0.000000e+00,ты хочешь остаться?,0.000000


## Observing transformer results

In [ ]:
top_10 = df_compare['Blue_Score_Transformer'].nlargest(10)
df_top_10 = df_compare[df_compare['Blue_Score_Transformer'].isin(top_10)]
df_top_10[['Original English','Actual Russian',"Transformer_Translation","Blue_Score_Transformer"]]

,Original English,Actual Russian,Transformer_Translation,Blue_Score_Transformer
0,I'd never hire Tom.,я бы никогда не взял тома на работу,я бы никогда не <unk> тома.,0.364093
25,I lived alone.,я жил один,я жил один.,0.240281
40,I'm sure Tom knows that.,я уверен что том это знает,"я уверен, что том это знает.",0.217119
42,Why don't you ask Tom how he did that?,почему бы тебе не спросить тома как он это сделал,"почему бы тебе не спросить тома, почему бы том...",0.267603
51,We didn't see any girls in the group.,мы не видели в группе девушек,мы не видели в <unk>,0.547518
61,Sometimes you surprise me.,иногда ты меня удивляешь,иногда ты меня <unk>,0.397635
65,Do you know where Tom bought his bicycle?,вы знаете где том купил свой велосипед,"ты знаешь, где том купил свой велосипед?",0.411134
72,We can't keep on fooling ourselves.,мы не можем и дальше заниматься,мы не можем плавать на <unk>,0.202052
83,That's not a crime.,это не преступление,это не преступление.,0.240281
91,"Does Tom play the mandolin, too?",том тоже играет на мандолине,том тоже играет в <unk>,0.265915


Right off the bat we see a major improvement compared to the NLST approach in our translations although the blue score is stil not the greatest, if we look at the sentences themselves, they are reather accurate.

* for instance "I lived alone."	--> "я жил один" | is a good translation. there's nothing to improve or change in fact this translation is correct and matches the actual russian word for word. (it added a perdiod at the end, but that does not really matter to much in converying meaning)

The transformer model succeeds very well in translating text consistently. the BLUE metric may not be the highest, due to punctuation and the unknown words... but the actual words themselves and the grammer/order they appear in is something the model succeeded in translating.


In [ ]:
bot_10 = df_compare['Blue_Score_Transformer'].nsmallest(10)
df_bot_10 = df_compare[df_compare['Blue_Score_Transformer'].isin(bot_10)]
df_bot_10[['Original English','Actual Russian',"Transformer_Translation","Blue_Score_Transformer"]]

,Original English,Actual Russian,Transformer_Translation,Blue_Score_Transformer
12,That fish is not edible.,эта рыба несъедобна,это не <unk>,0.0
15,The time women spend doing housework is now a ...,сейчас женщины тратят меньше времени на уборку...,теперь <unk> <unk> <unk> <unk> <unk>,0.0
28,The company shares give a high yield.,акции компании дают высокий доход,<unk> <unk> <unk> <unk> <unk> <unk> <unk>,0.0
31,You were cheating.,вы жульничали,ты был <unk>,0.0
33,The house is cold.,в доме холодно,дом <unk>,0.0
35,She sounded irritated.,в её голосе чувствовалось раздражение,она <unk>,0.0
45,Is eating healthy more expensive?,правильно питаться стоит дороже,<unk> — <unk> <unk>,0.0
54,The rear-view mirror fell off.,зеркало заднего вида,<unk> <unk> <unk> <unk> <unk> <unk>,0.0
59,The senate disapproved of the minister.,сенат осудил министра,<unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk...,0.0
66,How much is the rent for this room?,какова арендная плата за эту комнату,сколько стоит это <unk>,0.0


There is however, an elephant in the room, the russian vocabulary or the set of unique words our model had access to clearly was not large enough we have entire sentences comprised of unknown words at times... this problem can easily be solved by incorporating larger training datasets.

This could also be fixed also by changing the model to not predict \<unk\> as a russian word (target word), but rather for the placeholder for words not in the training set can be used as a context clue for what to put in that field....

for instance if I see the phrase: Hello my name is _________. and I've never seen or heard the word in the blank space (unknown/not in my training set). I can still infer that the person is introducing themselves and therfore I can assume that the blank will have a name in in, having a name in the blank space is alot closer to the true translation of the phrase than simply putting \
<blk\>.